# [이어 실행] 4.3절 목적함수 주입 — 저장된 모델은 불러오고 빠진 것만 학습

`런타임 > 모두 실행`만 누르면 된다. `crypto_objinj_retrain_fix1004.ipynb`의 셀을 그대로 불러와 실행하되,
학습 셀 3개 대신 "저장된 모델은 로드, 없는 것만 학습" 셀을 쓴다. 중간에 끊기면 다시 `모두 실행`하면 이어서 진행된다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 원본 노트북 셀 불러오기 + 설정 셀(1~7) 실행
import json
_SRC = '/content/drive/MyDrive/졸업프로젝트/rerun_fix1004b/crypto_objinj_retrain_fix1004.ipynb'
_nb = json.load(open(_SRC))
CELLS = [c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
         for c in _nb['cells'] if c['cell_type'] == 'code']
assert len(CELLS) == 17 and all(CELLS[i].startswith('%%time') for i in (8, 9, 10)), '원본 노트북 구조가 예상과 다름'
for i in range(1, 8):
    exec(CELLS[i], globals())
print('설정 완료')


In [ ]:
# 학습: 저장된 모델은 로드, 없는 것만 학습(원본과 같은 시드)
DIRS = {
    'base':   (MODEL_DIR / 'crypto_hrl_pool_v3_base_fix1004',            None,              4000),
    'objinj': (MODEL_DIR / 'crypto_hrl_pool_objinj_cash_fix1004',        AUX_LAMBDA,        5000),
    'strong': (MODEL_DIR / 'crypto_hrl_pool_objinj_cash_strong_fix1004', AUX_LAMBDA_STRONG, 6000),
}
pools = {k: {} for k in DIRS}
for key, (d, lam, seed_base) in DIRS.items():
    d.mkdir(parents=True, exist_ok=True)
    for pool_i, beta in enumerate(POOL_BETAS):
        p = d / f'agent_beta_{beta}.zip'
        if p.exists():
            pools[key][beta] = PPO.load(str(p))
            print(f'[{key}] β={beta}: 저장된 모델 로드')
            continue
        print(f'[{key}] β={beta}: 없음 → 학습 ({POOL_TIMESTEPS_EACH:,} 스텝, 약 15분)')
        rng = np.random.default_rng(seed_base + pool_i)
        if lam is None:
            env = ChunkedCashEnv(all_buckets=v3_train_buckets, chunks=seg_v3_train.chunks, beta=beta,
                                 risk_threshold=RISK_THRESHOLD, rng=rng, **env_kwargs)
        else:
            env = ObjectiveInjectedCashEnv(all_buckets=v3_train_buckets, chunks=seg_v3_train.chunks, beta=beta,
                                           risk_threshold=RISK_THRESHOLD, aux_lambda=lam, rng=rng, **env_kwargs)
        model = PPO('MlpPolicy', Monitor(env),
                    learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
                    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
                    max_grad_norm=0.5, policy_kwargs=dict(net_arch=[512, 256]),
                    verbose=0, device='cpu', seed=42)
        model.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)
        model.save(str(p))
        pools[key][beta] = model
        print(f'[{key}] β={beta}: 학습·저장 완료')

v3_baseline_models             = pools['base']
pool_models_objinj_cash        = pools['objinj']
pool_models_objinj_cash_strong = pools['strong']
assert all(len(v) == 4 for v in pools.values())
print('모델 12개 준비 완료')


In [ ]:
# 분석 셀(11~16) 실행 — 다양성 진단, valid/test 성과, 요약 CSV 저장
for i in range(11, 17):
    print(f'\n########## 원본 셀 {i} ##########')
    exec(CELLS[i], globals())
print('\n전부 완료')
